In [62]:
import pandas as pd
from dateutil.relativedelta import relativedelta

In [63]:
raw_df = pd.read_csv(r'data/undefind_DSMED.csv', sep=';')
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36297 entries, 0 to 36296
Data columns (total 13 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   ID истории болезни                    36297 non-null  object 
 1   Осн. диаг. при выписке МКБ10 (текст)  36297 non-null  object 
 2   Заголовок документа                   36297 non-null  object 
 3   Кол. лаб. показатель                  36236 non-null  object 
 4   Значение кол. показателя              36236 non-null  float64
 5   Ед. изм. кол. показателя              36236 non-null  object 
 6   Норма кол. показателя                 36236 non-null  object 
 7   Флаг нормы кол. показателя            36236 non-null  object 
 8   Кач. лаб. показатель                  7005 non-null   object 
 9   Значение кач. показателя              7005 non-null   object 
 10  Норма кач. показателя                 7005 non-null   object 
 11  Пол            

In [64]:
raw_df.nunique()

ID истории болезни                       243
Осн. диаг. при выписке МКБ10 (текст)      11
Заголовок документа                        1
Кол. лаб. показатель                     131
Значение кол. показателя                2608
Ед. изм. кол. показателя                  11
Норма кол. показателя                    123
Флаг нормы кол. показателя                 4
Кач. лаб. показатель                       7
Значение кач. показателя                 155
Норма кач. показателя                      5
Пол                                        2
Дата рождения пациента                   104
dtype: int64

In [65]:
raw_df['Кол. лаб. показатель'].value_counts()

Кол. лаб. показатель
Гемоглобин (HGB)                                        1114
Гематокрит (HCT)                                        1114
Средний объем эритроцита (MCV)                          1114
Среднее содержание гемоглобина в эритроците (MCH)       1114
Средняя концентрация гемоглобина в эритроците (MCHC)    1114
                                                        ... 
Базофилы, абсолютное количество                            1
Ширина распределения эритроцитов                           1
Средний объем тромбоцита                                   1
Ширина распределения тромбоцитов по объему                 1
Средний объём тромбоцитов                                  1
Name: count, Length: 131, dtype: int64

In [66]:
replacement_dict = {
    'wbc': ['Лейкоциты', 'Лейкоциты (WBC)', 'Общее количество лейкоцитов (WBC)', 'WBC'],
    'rbc': ['Эритроциты', 'Эритроциты (RBC)', 'Общее количество эритроцитов (RBC)', 'RBC'],
    'hgb': ['Гемоглобин', 'Гемоглобин (HGB)', 'HGB'],
    'hct': ['Гематокрит', 'Гематокрит (HCT)', 'HCT'],
    'mcv': ['Средний объем эритроцита', 'Средний объем эритроцита (MCV)', 'MCV'],
    'mch': ['Среднее содержание гемоглобина в эритроците', 'Среднее содержание гемоглобина в эритроците (MCH)', 'MCH'],
    'mchc': ['Средняя концентрация гемоглобина в эритроците', 'Средняя концентрация гемоглобина в эритроците (MCHC)', 'MCHC'],
    'plt': ['Тромбоциты', 'Тромбоциты (PLT)', 'PLT'],
    'pct': ['Тромбокрит', 'Тромбокрит (PCT)', 'PCT'],
    'mpv': ['Средний объем тромбоцита', 'Средний объём тромбоцитов', 'Средний объем тромбоцита (MPV)', 'MPV (Средний объём тромбоцитов)', 'MPV ', 'MPV'],
    'pdw': ['Ширина распределения тромбоцитов', 'Ширина распределения тромбоцитов по объему', 'Ширина распределения тромбоцитов (PDW)', 'PDW'],
    'rdw': ['Ширина распределения эритроцитов', 'Ширина распределения эритроцитов по объему (RDW)', 'RDW', 'Ширина распределения эритроцитов (RDW)'],
    'rdv_cv': ['Ширина распределения эритроцитов по объему, коэффициент вариации (RDW-CV)', 'Ширина распределения эритроцитов по объему, коэффициент вариации', 'RDW-CV '],
    'rdv_sd': ['Ширина распределения эритроцитов по объему, стандартное отклонение (RDW-SD)','Ширина распределения эритроцитов по объему, стандартное отклонение','Ширина распределения эритроцитов, стандартное отклонение (RDW-SD)', 'RDW-SD'],
    'ne_abs': ['Нейтрофилы, абсолютное количество', 'Нейтрофилы, абсолютное количество (NE#)', 'Абсолютное количество нейтрофилов (NE#)', 'Нейтрофилы #'],
    'ne_rel': ['Нейтрофилы, относительное количество', 'Нейтрофилы, относительное количество (NE%)', 'Относительное количество нейтрофилов (NE%)', 'Нейтрофилы %', 'NE%'],
    'ly_abs': ['Лимфоциты, абсолютное количество', 'Лимфоциты, абсолютное количество (LY#)', 'Лимфоциты', 'Абсолютное количество лимфоцитов (LY#)', 'Лимфоциты #'],
    'ly_rel': ['Лимфоциты, относительное количество', 'Лимфоциты, относительное количество (LY%)', 'Лимфоциты %', 'Относительное количество лимфоцитов (LY%)', 'Лимфоциты %', 'LY%'],
    'mo_abs': ['Моноциты, абсолютное количество', 'Моноциты, абсолютное количество (MO#)', 'Моноциты', 'Абсолютное количество моноцитов (MO#)', 'Моноциты #'],
    'mo_rel': ['Моноциты, относительное количество', 'Моноциты, относительное количество (MO%)', 'Моноциты %', 'Моноциты %', 'Относительное количество моноцитов (MO%)', 'MO%'],
    'eo_abs': ['Эозинофилы, абсолютное количество', 'Эозинофилы, абсолютное количество (EO#)', 'Эозинофилы', 'Абсолютное количество эозинофилов (EO#)', 'Эозинофилы # '],
    'eo_rel': ['Эозинофилы, относительное количество', 'Эозинофилы, относительное количество (EO%)', 'Эозинофилы %', 'Относительное количество эозинофилов (EO%)', 'EO%'],
    'ba_abs': ['Базофилы, абсолютное количество', 'Базофилы, абсолютное количество (BA#)', 'Базофилы', 'Абсолютное количество базофилов (BA#)', 'Базофилы #', 'Базофилы # '],
    'ba_rel': ['Базофилы, относительное количество', 'Базофилы, относительное количество (BA%)', 'Базофилы %', 'Относительное количество базофилов (BA%)', 'BA%'],
    'mxd_abs': ['Смешанная фракция, абсолютное количество (MXD#)', 'MXD# ', 'Смешанная фракция, абсолютное количество', 'MXD#', 'MXD'],
    'mxd_rel': ['Смешанная фракция, относительное количество (MXD%)', 'MXD%', 'Смешанная фракция, относительное количество'],
    'soe_v': ['СОЭ Вест.', 'Скорость оседания эритроцитов (СОЭ) по Вестергрену'],
    'soe_p': ['СОЭ по Панченкову', 'СОЭ Панч.'],
    'cp': ['Цветовой показатель'],
    'pal': ['Палочкоядерные'],
    'seg': ['Сегментоядерные'],
    'plasma': ['Плазматические клетки', 'Плазматич. клетки'],
    'myelo': ['Миелоциты'],
    'yunye': ['Юные'],
    'blasty': ['Бласты'],
    'normobl_abs': ['Нормобласты', 'Нормобласты #'],
    'normobl_rel': ['Нормобласты %'],
    'ret_rel': ['RET%', 'Ретикулоциты %'],
    'ret_abs': ['Ретикулоциты кол-во'],
    'plcr': ['P-LCR'],
    'noclass_abs': ['Неклассифицируемые кол-во'],
    'noclass_rel': ['Неклассифицируемые %'],
    'prolym': ['Пролимфоциты'],
    'promyelo': ['Промиелоциты']
}

In [67]:
prep_df = raw_df.copy()
prep_df = prep_df.dropna(subset=['Кол. лаб. показатель'])
prep_df['Дата рождения пациента'] = pd.to_datetime(prep_df['Дата рождения пациента'])
prep_df['lab_test_date'] = pd.to_datetime('2025-07-21')
prep_df['age'] = prep_df.apply(lambda row: relativedelta(row['lab_test_date'], row['Дата рождения пациента']).years, axis=1)
prep_df['gender'] = prep_df['Пол'].apply(lambda x: 0 if 'ж' in x.lower() else 1)


In [68]:
# Функция для замены значения на ключ словаря
def replace_with_dict_key(value, rep_dict):
    for key, values_list in rep_dict.items():
        if value in values_list:
            return key
    return None  # Если значение не найдено в словаре, оставляем пустым

# Применяем функцию к столбцу
prep_df['lab_test'] = prep_df['Кол. лаб. показатель'].apply(lambda x: replace_with_dict_key(x, replacement_dict))

In [69]:
prep_df['lab_test'].value_counts()

lab_test
ly_abs         2213
rbc            1619
hgb            1619
hct            1619
mcv            1619
mch            1619
mchc           1619
plt            1617
wbc            1616
mo_abs         1518
ly_rel         1470
eo_abs         1388
ne_rel         1332
ne_abs         1319
ba_abs         1087
mpv             982
pct             925
cp              832
rdw             829
eo_rel          824
ba_rel          822
mo_rel          815
seg             740
pal             736
rdv_sd          698
rdv_cv          696
pdw             657
soe_v           568
mxd_abs         566
mxd_rel         468
myelo           316
yunye           304
noclass_rel     239
soe_p           226
normobl_abs     192
blasty          143
noclass_abs     117
plcr             81
ret_rel          57
ret_abs          44
plasma           30
promyelo         21
normobl_rel      18
prolym           16
Name: count, dtype: int64

In [70]:
prep_df['Флаг нормы кол. показателя'].value_counts(dropna=False)

Флаг нормы кол. показателя
Норм     21218
Пониж     8207
Повыш     6748
Норма       63
Name: count, dtype: int64

In [71]:
prep_df['result'] = prep_df['Флаг нормы кол. показателя'].apply(lambda x: 0 if 'норм' in x.lower()
                                                                else (-1 if "пониж" in x.lower()
                                                                      else (1 if "повыш" in x.lower()
                                                                            else x)))

In [72]:
pasp_df = prep_df[['ID истории болезни', 'gender', 'age']].drop_duplicates().copy()
pasp_df

,ID истории болезни,gender,age
0,2e1d0b3f-488a-11ed-ab5a-0050568844e6,0,62
154,24612d4d-c466-11ec-ab54-0050568844e6,0,86
430,b0a85bb2-3404-11ed-ab56-0050568844e6,0,62
433,09564dae-ebb9-11ec-ab56-0050568844e6,0,78
461,2efd978b-cd3d-11ed-8604-005056880ecb,1,67
...,...,...,...
35717,69714728-b691-11ee-8606-005056880ecb,1,68
35803,ffdf0ad3-bb5b-11ee-ab6f-0050568844e6,1,71
35946,20f82885-b5cf-11ee-ab6f-0050568844e6,1,77
36146,6166e156-c1af-11ed-8602-005056880ecb,1,43


In [73]:
ref_df = prep_df[['Пол', 'lab_test', 'Норма кол. показателя']].drop_duplicates()
ref_df[['lo_ref', 'hi_ref']] = ref_df['Норма кол. показателя'].str.split(':', n=1, expand=True)
ref_df.drop(columns=['Норма кол. показателя'], inplace=True)
ref_df['lo_ref'] = ref_df['lo_ref'].str.replace(',', '.')
ref_df['hi_ref'] = ref_df['hi_ref'].str.replace(',', '.')
ref_df.dropna(inplace=True)
ref_df[['lo_ref', 'hi_ref']] = ref_df[['lo_ref', 'hi_ref']].astype('float')
ref_df.drop_duplicates(inplace=True)
ref_df.sort_values(['Пол', 'lab_test'])

,Пол,lab_test,lo_ref,hi_ref
37,Ж,ba_abs,0.0,0.10
129,Ж,ba_abs,0.0,2.00
32,Ж,ba_rel,0.0,2.00
308,Ж,blasty,0.0,0.00
38,Ж,cp,0.8,1.05
...,...,...,...,...
6240,М,soe_p,2.0,20.00
484,М,soe_v,2.0,20.00
462,М,wbc,4.0,11.00
11344,М,wbc,4.0,8.80


In [106]:
mask = prep_df['Кач. лаб. показатель'].notna() & prep_df['Кач. лаб. показатель'].ne('Комментарий')
real_qual_df = prep_df[mask][['ID истории болезни', 'Кач. лаб. показатель', 'Значение кач. показателя', 'Норма кач. показателя']].copy()
real_qual_df

,ID истории болезни,Кач. лаб. показатель,Значение кач. показателя,Норма кач. показателя
930,76416b20-b70b-11ec-ab54-0050568844e6,Анизоцитоз,+,+-
931,76416b20-b70b-11ec-ab54-0050568844e6,Гипохромия,+,+-
932,76416b20-b70b-11ec-ab54-0050568844e6,Анизоцитоз,+,+-
933,76416b20-b70b-11ec-ab54-0050568844e6,Гипохромия,+,+-
934,76416b20-b70b-11ec-ab54-0050568844e6,Анизоцитоз,+,+-
...,...,...,...,...
36037,20f82885-b5cf-11ee-ab6f-0050568844e6,Пойкилоцитоз,+++,+-
36039,20f82885-b5cf-11ee-ab6f-0050568844e6,Анизоцитоз,+++,+-
36040,20f82885-b5cf-11ee-ab6f-0050568844e6,Пойкилоцитоз,+++,+-
36042,20f82885-b5cf-11ee-ab6f-0050568844e6,Анизоцитоз,+++,+-


In [107]:
real_qual_df['Норма кач. показателя'].value_counts(dropna=False)

Норма кач. показателя
+-             2170
Общая норма     223
0:1              29
0:2              20
Name: count, dtype: int64

In [108]:
real_qual_df['Значение кач. показателя'].value_counts(dropna=False)

Значение кач. показателя
+            1250
++            895
+++           182
+-             66
3.0:100.0      29
3:100          20
Name: count, dtype: int64

In [109]:
real_qual_df.groupby(['Кач. лаб. показатель', 'Норма кач. показателя']).count()

ID истории болезни  \
Кач. лаб. показатель Норма кач. показателя                       
Анизоцитоз           +-                                    910   
                     Общая норма                           113   
Гипохромия           +-                                    191   
                     Общая норма                            54   
Макроцитоз           +-                                    141   
Микроцитоз           +-                                    139   
                     Общая норма                            29   
Нормобласты          0:1                                    29   
                     0:2                                    20   
Пойкилоцитоз         +-                                    789   
                     Общая норма                            27   

                                            Значение кач. показателя  
Кач. лаб. показатель Норма кач. показателя                            
Анизоцитоз           +-                                          910  
                     Общая норма                                 113  
Гипохромия           +-                                          191  
                     Общая норма                                  54  
Макроцитоз           +-                                          141  
Микроцитоз           +-                                          139  
                     Общая норма                                  29  
Нормобласты          0:1                                          29  
                     0:2                                          20  
Пойкилоцитоз         +-                                          789  
                     Общая норма                                  27

In [110]:
real_result_dict = {
    0: ['+', '-', '+-'],
    1: ['++', '3.0:100.0', '3:100'],
    2: ['+++']
}

real_qual_df['Значение кач. показателя'] = real_qual_df['Значение кач. показателя'].apply(lambda x: replace_with_dict_key(x, real_result_dict))
real_qual_df.drop(columns=['Норма кач. показателя'], inplace=True)
real_qual_df.drop_duplicates()

,ID истории болезни,Кач. лаб. показатель,Значение кач. показателя
930,76416b20-b70b-11ec-ab54-0050568844e6,Анизоцитоз,0
931,76416b20-b70b-11ec-ab54-0050568844e6,Гипохромия,0
994,76416b20-b70b-11ec-ab54-0050568844e6,Анизоцитоз,1
995,76416b20-b70b-11ec-ab54-0050568844e6,Пойкилоцитоз,1
1758,86475e95-6a40-11ed-ab5f-0050568844e6,Пойкилоцитоз,1
...,...,...,...
34474,69f13457-7020-11ef-ab70-0050568844e6,Пойкилоцитоз,1
34770,e8e7f3c5-61bb-11ee-ab6d-0050568844e6,Анизоцитоз,2
34771,e8e7f3c5-61bb-11ee-ab6d-0050568844e6,Пойкилоцитоз,2
35946,20f82885-b5cf-11ee-ab6f-0050568844e6,Анизоцитоз,2


In [117]:
real_qual_df.pivot_table(index='ID истории болезни', columns='Кач. лаб. показатель', values='Значение кач. показателя', aggfunc='first')

Кач. лаб. показатель,Анизоцитоз,Гипохромия,Макроцитоз,Микроцитоз,Нормобласты,Пойкилоцитоз
ID истории болезни,,,,,,
06a83aaa-8e27-11ec-ab52-0050568844e6,0.0,0.0,NaN,NaN,NaN,0.0
08475dc2-40af-11ee-8604-005056880ecb,0.0,NaN,NaN,NaN,NaN,0.0
0c2c320c-e69d-11ea-80d7-901b0e63368b,0.0,NaN,NaN,0.0,NaN,NaN
0e99ec58-c708-11ed-8602-005056880ecb,0.0,NaN,NaN,NaN,NaN,0.0
139dbc03-3277-11ed-ab56-0050568844e6,0.0,0.0,NaN,NaN,NaN,NaN
1c63c1e6-6b70-11eb-80ea-901b0e63368a,2.0,NaN,NaN,NaN,NaN,NaN
1e2e8fda-be65-11ed-8600-005056880ecb,1.0,NaN,NaN,NaN,NaN,1.0
1e9f2b4d-98af-11e9-80e1-901b0e63368a,NaN,NaN,NaN,NaN,1.0,NaN
20f82885-b5cf-11ee-ab6f-0050568844e6,2.0,NaN,NaN,NaN,NaN,2.0


In [118]:
work_df = prep_df.pivot_table(values='Значение кол. показателя', columns='lab_test', index='ID истории болезни', aggfunc='first')
work_df = work_df.merge(pasp_df, on='ID истории болезни', how='inner')

In [116]:
prep_df.pivot_table(values='result', columns='lab_test', index='ID истории болезни', aggfunc='first')

lab_test,ba_abs,ba_rel,blasty,cp,eo_abs,eo_rel,hct,hgb,ly_abs,ly_rel,...,rdv_cv,rdv_sd,rdw,ret_abs,ret_rel,seg,soe_p,soe_v,wbc,yunye
ID истории болезни,,,,,,,,,,,,,,,,,,,,,
05deb1af-cf40-11e9-80ba-901b0e633689,0.0,0.0,NaN,1.0,0.0,0.0,-1.0,-1.0,0.0,-1.0,...,NaN,NaN,1.0,NaN,1.0,0.0,0.0,NaN,0.0,1.0
06a83aaa-8e27-11ec-ab52-0050568844e6,1.0,0.0,NaN,-1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,NaN,0.0,NaN,0.0,1.0,1.0
0839c9c1-1109-11ef-8607-005056880ecb,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,...,0.0,0.0,1.0,NaN,NaN,0.0,NaN,1.0,0.0,1.0
08475dc2-40af-11ee-8604-005056880ecb,1.0,0.0,NaN,0.0,0.0,0.0,-1.0,-1.0,0.0,0.0,...,1.0,1.0,1.0,NaN,NaN,0.0,NaN,0.0,0.0,NaN
087489cd-0fb8-11ec-bb98-2cea7fe73f75,0.0,0.0,NaN,0.0,0.0,0.0,-1.0,-1.0,0.0,0.0,...,0.0,0.0,0.0,NaN,NaN,0.0,1.0,NaN,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fc6b2aea-d5ad-11ec-ab54-0050568844e6,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,NaN,NaN,1.0,NaN,0.0,0.0,NaN
fc865ace-f83b-11ec-ab56-0050568844e6,0.0,0.0,NaN,-1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,NaN,0.0,NaN,0.0,0.0,NaN
ff883bc4-80a2-11eb-bb94-2cea7fe73f75,1.0,1.0,1.0,0.0,0.0,0.0,-1.0,-1.0,0.0,-1.0,...,NaN,1.0,1.0,NaN,NaN,0.0,0.0,NaN,0.0,1.0


In [113]:
def check_data_quality(data,
                       freq_treshold=0.95,
                       nunique_threshold=0.95,
                       null_threshold=0):
    """Проводит поиск дублей, пропусков и неинформативных признаков и выводит результат в консоль.
    
    Неинформативным считается признак с долей уникальных значений или повторов выше установленного порога.
    
    Parameters
    ----------
        data : DataFrame
            Датафрейм для анализа
        freq_treshold : float
            Порог масимальной частоты встречаемости признака
        nunique_treshhold : float
            Порог максимальной уникальности признака
        null_threshold : float
            Порог максимального отстутсвия признака
    """
    
    # Поиск дублей
    try:
        print(f'Число найденных дублей: {
            data.duplicated().value_counts().loc[True]
            }\n')
    except KeyError:
        print('Количество дублей: 0\n')

    # Поиск пропущенных значений
    cols_null_percent = data.isnull().mean()
    cols_with_null = cols_null_percent[cols_null_percent > null_threshold]\
        .sort_values(ascending=False)
    if cols_with_null.shape[0]: display(cols_with_null)
    print(f'Количество признаков с пустыми значениями: {cols_with_null.shape[0]}\n')

    # Поиск неинформативных признаков
    low_information_cols = []
    bad_feat_flag = False
    # цикл по всем столбцам
    for col in data.columns:
        #наибольшая относительная частота в признаке
        top_freq = data[col].value_counts(normalize=True).max()
        #доля уникальных значений от размера признака
        nunique_ratio = data[col].nunique() / data[col].count()
        # сравниваем наибольшую частоту с порогом
        if top_freq > freq_treshold:
            bad_feat_flag = True
            low_information_cols.append(col)
            print(f'{col}: {top_freq:.2%}% одинаковых значений')
        # сравниваем долю уникальных значений с порогом
        if nunique_ratio > nunique_threshold:
            bad_feat_flag = True
            low_information_cols.append(col)
            print(f'{col}: {nunique_ratio:.2%} уникальных значений')
    if not bad_feat_flag:
        print('Неинформативных признаков не найдено.')

In [114]:
print('Проверка тренировочной выборки:'+'\n'+'-' * 30)
check_data_quality(work_df)

Проверка тренировочной выборки:
------------------------------
Количество дублей: 0



normobl_rel    0.962810
prolym         0.950413
promyelo       0.946281
plasma         0.942149
plcr           0.938017
ret_abs        0.880165
ret_rel        0.847107
noclass_abs    0.776860
blasty         0.735537
soe_p          0.714876
noclass_rel    0.669421
normobl_abs    0.619835
mxd_rel        0.483471
myelo          0.471074
rdv_cv         0.438017
pdw            0.429752
yunye          0.429752
mxd_abs        0.425620
rdv_sd         0.425620
pct            0.301653
soe_v          0.289256
mpv            0.264463
ba_abs         0.037190
pal            0.016529
seg            0.012397
eo_rel         0.008264
eo_abs         0.008264
ba_rel         0.004132
mo_rel         0.004132
rdw            0.004132
mo_abs         0.004132
cp             0.004132
dtype: float64

Количество признаков с пустыми значениями: 32

ID истории болезни: 100.00% уникальных значений
ly_abs: 99.59% уникальных значений
ly_rel: 97.11% уникальных значений
mchc: 96.28% уникальных значений
mcv: 95.45% уникальных значений
mo_abs: 96.27% уникальных значений
mxd_rel: 96.80% уникальных значений
ne_abs: 98.76% уникальных значений
ne_rel: 98.35% уникальных значений
normobl_rel: 100.00%% одинаковых значений
plt: 98.35% уникальных значений
rbc: 96.28% уникальных значений
ret_abs: 100.00% уникальных значений
ret_rel: 97.30% уникальных значений
wbc: 97.93% уникальных значений
